# 04 · E4: el LLM como anotador (etiquetas *silver* + pocas etiquetas *gold*)

### Qué es este experimento

E1 mostró que un modelo entrenado con **miles** de oraciones anotadas por médicos funciona muy bien (F1 ≈ 0,80 en enfermedades). E2 mostró que un LLM **sin entrenar** queda bastante más abajo (≈ 0,66). Pero en la práctica, en un CESFAM, nadie tiene miles de fichas anotadas.

E4 combina lo mejor de ambos: el **LLM anota** texto sin anotar (etiquetas *silver*, "de plata") y con esas etiquetas se **entrena un modelo como el de E1**. Agregamos además un esfuerzo humano pequeño: 0, 10, 50 o 100 oraciones con anotación real (*gold*, "de oro").

**Analogía:** un médico general (el LLM) revisa 2.000 fichas y marca lo que cree que son enfermedades, con errores. Un especialista revisa bien solo 50. Con ese material se forma a un interno (el encoder). La pregunta es si el interno termina sabiendo más que el médico general que le enseñó.

### De dónde salen los ejemplos "humanos"

No se anota nada nuevo. Son anotaciones **reales del corpus**, hechas por médicos y dentistas, pero usamos solo unas pocas a propósito: simulamos que solo existen 0, 10, 50 o 100. El test sigue siendo el gold completo, sin tocar.

### Qué comparamos

| Sistema | Cómo se entrena |
|---|---|
| **Silver + gold** (E4) | 2.000 oraciones anotadas por el LLM + N oraciones gold |
| **Solo gold** | Solo las N oraciones gold (sin LLM) |
| **E1** (referencia) | Las 8.014 oraciones gold |
| **E2** (referencia) | El LLM directamente, sin entrenar nada |

### Conexión con el curso

- **U4 · Modelos híbridos:** un LLM (*decoder*) enseña a un *encoder*.
- **U8 · IA generativa:** el LLM genera datos de entrenamiento sintéticos (etiquetas *silver*).
- **U9 · Adaptación de modelos fundacionales:** destilación de conocimiento de un modelo grande a uno pequeño y rápido.
- **U2–U3:** el ajuste fino del encoder es el mismo de E1.

In [ ]:
from pathlib import Path
import sys, json, time, pickle
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import yaml
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import asdict
from datos.cwlc import cargar_split
from modelos import e2_llm as E2
from modelos import e4_destilacion as E4
from modelos.e1_encoder import Config, entrenar, predecir, evaluar, guardar_resultados, dispositivo
from reproducibilidad import huella_completa, verificar_huellas_datos, agregar_semillas, sha256_bytes
from sistema import mantener_despierto, dejar_dormir

cfg_global = yaml.safe_load(open(RAIZ / "configs" / "experimentos.yaml"))
cfg4 = cfg_global["destilacion"]
carpeta = RAIZ / "data" / "raw" / "cwlc"
train, dev, test = (cargar_split(carpeta, s) for s in ["train", "dev", "test"])
ok, difs = verificar_huellas_datos(carpeta, "cwlc")
assert ok, f"¡Los datos cambiaron! {difs}"
print(f"train={len(train)}  dev={len(dev)}  test={len(test)} · huellas verificadas · anotador: {cfg4['anotador']}")

## 1. Diseño y cuidados contra la fuga

- **Gold anidado:** las 10 oraciones gold están dentro de las 50, y estas dentro de las 100. Así, la curva mide el efecto de *agregar* anotaciones, no de cambiar cuáles.
- **Silver disjunto:** las 2.000 oraciones que anota el LLM nunca coinciden con las gold, y son las mismas para todos los N.
- **Ejemplos para el LLM solo desde el gold disponible.** En E2 el *few-shot* dinámico buscaba ejemplos en todo train. Aquí eso sería trampa, porque usaría anotaciones que "no existen". Con N = 0 el LLM trabaja sin ejemplos.
- **Sin dev para elegir la época.** En el escenario simulado no hay 890 oraciones de dev anotadas. Se entrena un número fijo de épocas y se usa la última. E1 sí usó dev, lo que le da una pequeña ventaja.
- **Encoder:** `bsc-bio-ehr-es` (español), porque no tiene el riesgo de haber visto el test.
- **Semillas:** cada entrenamiento se repite con 3 semillas, como en E1. El gold y el silver son fijos (semilla 42).

## 2. Configuración

- **`MODO = "rapido"`**: 200 oraciones silver, solo Enfermedad, 1 semilla (~45 min). Sirve para revisar que todo funcione.
- **`MODO = "completo"`**: 2.000 oraciones silver, las 3 entidades, 3 semillas. Toma varias horas: ~3,5 h de anotación con el LLM + ~3 h de entrenamientos.

In [ ]:
MODO = "rapido"          # "rapido" para probar | "completo" para reportar

NS_GOLD = cfg4["gold_humano"]                               # [0, 10, 50, 100]
ENCODER = cfg_global["encoders"][cfg4["encoder"]]
ent = cfg_global["entrenamiento_encoder"]
if MODO == "rapido":
    SILVER_N, TIPOS, SEMILLAS = 200, ["Disease"], [42]
else:
    SILVER_N, TIPOS, SEMILLAS = cfg4["silver_n"], cfg_global["entidades"], cfg_global["semillas"]
EPOCAS, PASOS_MIN = cfg4["epocas"], cfg4["pasos_min"]

pool = E4.pool_silver(train, SILVER_N)
SALIDA = RAIZ / "results" / f"e4_{MODO}"
CACHE = RAIZ / "results" / "cache" / f"e4_{MODO}"
n_entren = len(NS_GOLD) * len(TIPOS) * len(SEMILLAS) + sum(n > 0 for n in NS_GOLD) * len(TIPOS) * len(SEMILLAS)
print(f"{MODO}: {SILVER_N} oraciones silver × {len(NS_GOLD)} niveles de gold = {SILVER_N * len(NS_GOLD)} llamadas al LLM; "
      f"{n_entren} entrenamientos")

## 3. Paso 1: el LLM anota las oraciones silver

Para cada nivel de gold (0, 10, 50, 100), el LLM anota las mismas oraciones silver, con ejemplos sacados solo de ese gold. Como **sí** conocemos la anotación real de esas oraciones (solo la escondimos), podemos medir **qué tan buen "maestro" es el LLM**: el F1 de sus etiquetas silver.

In [ ]:
despierto = mantener_despierto()
silvers, calidad = {}, []
for n in NS_GOLD:
    gold = E4.seleccionar_gold(train, n)
    cache = E2.Cache(CACHE / f"silver_n{n}.jsonl")
    print(f"\n=== Anotando {len(pool)} oraciones silver con gold N = {n} ===")
    resp = E2.correr(cfg4["anotador"], pool, E4.selector_para(gold, cfg_global["k_ejemplos"]), cache, cada=200)
    silver, ev = E4.oraciones_silver(pool, resp)
    silvers[n] = silver
    q = E4.calidad_silver(pool, silver)
    for tipo, m in q.items():
        calidad.append({"N gold": n, "tipo": tipo, **{k: round(v, 3) for k, v in m.items()},
                        "alucinación": round(ev["tasa_alucinacion"], 3), "JSON inválidos": ev["json_invalidos"]})
calidad = pd.DataFrame(calidad)
calidad.pivot(index="N gold", columns="tipo", values="f1")

## 4. Paso 2: entrenar los encoders

Para cada nivel de gold, cada entidad y cada semilla se entrenan dos modelos:

- **silver + gold:** las oraciones silver con las etiquetas del LLM, más las N gold con su etiqueta real.
- **solo gold:** solo las N oraciones gold. Con N = 0 no hay nada que entrenar.

**Mismo esfuerzo de entrenamiento para todos.** Una *época* es una pasada por los datos, y en cada pasada los pesos se actualizan una vez por lote de 16 oraciones. Con 10 oraciones, una época es **una sola** actualización; con 2.000, son 125. Si diéramos el mismo número de épocas, los modelos chicos casi no alcanzarían a aprender y la comparación sería injusta (la primera prueba rápida mostró justo eso: F1 = 0). Por eso todos los modelos reciben **al menos 300 actualizaciones**: si sus datos son pocos, se repiten más veces. Es como dar a todos los alumnos las mismas horas de estudio, aunque unos tengan una guía de 10 ejercicios y otros de 2.000.

Igual que en E1, si un resultado ya existe se salta, así que se puede interrumpir y retomar.

In [ ]:
for n in NS_GOLD:
    gold = E4.seleccionar_gold(train, n)
    corpus = {"silver_gold": silvers[n] + gold}
    if n > 0:
        corpus["solo_gold"] = gold
    for condicion, datos in corpus.items():
        for tipo in TIPOS:
            for semilla in SEMILLAS:
                ruta = SALIDA / f"{condicion}_n{n}_{tipo}_s{semilla}.json"
                if ruta.exists():
                    print(f"(ya existe) {ruta.name}"); continue
                print(f"\n=== {condicion} · N gold = {n} · {tipo} · semilla {semilla} ({len(datos)} oraciones) ===")
                cfg = Config(modelo=ENCODER, tipo=tipo, lr=ent["lr"], batch=ent["batch"], max_len=ent["max_len"], semilla=semilla,
                             epocas=EPOCAS, pasos_min=PASOS_MIN, seleccion="ultima")
                t0 = time.time()
                modelo, cod, historial = entrenar(cfg, datos, dev)
                pred = predecir(modelo, cod, test, tipo, disp=dispositivo())
                gold_test = [e for o in test for e in o.entidades if e[3] == tipo]
                res = evaluar(gold_test, pred, test, tipo, train=train)
                res.update({"condicion": condicion, "n_gold": n, "n_silver": len(pool) if condicion == "silver_gold" else 0,
                            "minutos_total": round((time.time() - t0) / 60, 1)})
                config = asdict(cfg) | {"anotador": cfg4["anotador"], "guia": sha256_bytes(E2.GUIA.encode()),
                                        "gold_ids": [o.id for o in gold], "silver_ids": sha256_bytes(json.dumps([o.id for o in pool]).encode())}
                huella = huella_completa(config, carpeta, modelo=ENCODER)
                huella["ollama"] = E2.huella_ollama(cfg4["anotador"])
                guardar_resultados(ruta, cfg, historial, res, huella)
                (CACHE / f"{ruta.stem}.pkl").write_bytes(pickle.dumps(pred))
                print(f"  F1 estricto test = {res['estricto']['f1']:.3f}")
                del modelo
dejar_dormir(despierto)

## 5. Resultados

La tabla junta, para cada entidad y nivel de gold, el F1 estricto en test (media ± DE de las semillas) de los modelos E4, junto con la calidad del "maestro" (las etiquetas del LLM medidas contra la anotación real de las oraciones silver).

In [ ]:
filas = []
for f in sorted(SALIDA.glob("*.json")):
    d = json.loads(f.read_text()); r = d["resultados"]
    filas.append({"condición": r["condicion"], "N gold": r["n_gold"], "tipo": d["config"]["tipo"], "semilla": d["config"]["semilla"],
                  "F1 estricto": r["estricto"]["f1"], "F1 laxo": r["laxo"]["f1"], "P": r["estricto"]["precision"], "R": r["estricto"]["recall"],
                  "recall nuevas": r["menciones_nuevas"]["recall_estricto"], "min": r["minutos_total"]})
e4 = pd.DataFrame(filas)
def media_de(x):
    a = agregar_semillas(list(x)); return f"{a['media']:.3f} ± {a['de']:.3f}" if a["n"] > 1 else f"{a['media']:.3f}"
tabla = e4.groupby(["tipo", "N gold", "condición"])[["F1 estricto", "F1 laxo", "P", "R", "recall nuevas"]].agg(media_de).unstack("condición")
display(tabla)
display(calidad.pivot(index="N gold", columns="tipo", values="f1").rename(columns=lambda t: f"maestro {t}"))

In [ ]:
# Referencias: E1 (todo el gold, encoder español) y E2 (el LLM directo en el test)
ref = {}
for tipo in TIPOS:
    e1 = [json.loads(p.read_text())["resultados"]["estricto"]["f1"] for p in (RAIZ / "results" / "e1_completo").glob(f"espanol_clinico_{tipo}_s*.json")]
    e2 = {}
    for est in ["zero_shot", "few_shot_dinamico"]:
        p = RAIZ / "results" / "e2_completo" / f"{cfg4['anotador'].replace(':', '-')}_{est}.json"
        if p.exists():
            e2[est] = json.loads(p.read_text())["resultados"]["por_tipo"][tipo]["estricto"]["f1"]
    ref[tipo] = {"E1": sum(e1) / len(e1) if e1 else None, **e2}

fig, axes = plt.subplots(1, len(TIPOS), figsize=(5 * len(TIPOS), 4), squeeze=False)
for ax, tipo in zip(axes[0], TIPOS):
    for condicion, estilo in [("silver_gold", "o-"), ("solo_gold", "s--")]:
        g = e4[(e4["tipo"] == tipo) & (e4["condición"] == condicion)].groupby("N gold")["F1 estricto"]
        if len(g.mean()):
            ax.errorbar(g.mean().index, g.mean(), yerr=g.std().fillna(0), fmt=estilo, capsize=3,
                        label="silver + gold (E4)" if condicion == "silver_gold" else "solo gold")
    if ref[tipo]["E1"]:
        ax.axhline(ref[tipo]["E1"], ls=":", c="k", label="E1 (8.014 gold)")
    if "few_shot_dinamico" in ref[tipo]:
        ax.axhline(ref[tipo]["few_shot_dinamico"], ls="-.", c="gray", label="E2 LLM directo (dinámico)")
    ax.set_xscale("symlog", linthresh=10); ax.set_xticks(NS_GOLD); ax.set_xticklabels(NS_GOLD)
    ax.set_xlabel("oraciones gold disponibles"); ax.set_ylabel("F1 estricto (test)"); ax.set_ylim(0, 1); ax.set_title(tipo)
axes[0][0].legend(fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()

## 6. Conclusiones (completar después de la corrida)

- ¿El "alumno" (encoder entrenado con silver) supera al "maestro" (el LLM)? Comparar su F1 con la calidad del silver y con E2.
- ¿Cuánto aporta el silver frente a entrenar **solo** con las N oraciones gold?
- ¿Cuántas oraciones gold hacen falta para acercarse a E1? ¿Dónde se aplana la curva?
- ¿Qué entidad se beneficia más? (Abreviatura era la más difícil para el LLM.)
- Costo: horas de LLM + horas de entrenamiento frente a anotar a mano.